# 04. 규칙 기반 모듈형 모델

`MODIS ≈ Σ(낮·밤) [ Σ(FRP 구간) 계수 × VIIRS 탐지 수 + 이웃 계수 × 이웃 8칸 VIIRS 탐지 수 ]` — 모든 계수를 표로 읽을 수 있는 모델입니다.

| 모듈 | 규칙 |
| --- | --- |
| 1. 고정 열원 제거 | `type 0`(식생 화재)만 집계. 과거 데이터에만 있는 열이라 7일 샘플에는 적용되지 않습니다 |
| 2. 낮·밤 분리 | 낮과 밤을 따로 환산 |
| 3. 작은 불 보정 | FRP 구간(<2, 2–5, 5–20, ≥20 MW)마다 다른 계수 |
| 4. 주변 칸 | VIIRS는 못 봤는데 MODIS가 잡은 불(관측 시각·픽셀 경계 차이)을 이웃 8칸의 VIIRS 탐지로 설명 |
| 5. 지역·계절 | 위도대 × 계절별 계수. 남반구는 계절을 6개월 밀어서 맞춥니다 |
| 6. 희소 그룹 | 데이터가 적은 그룹은 상위 그룹(위도대 → 전 세계)의 계수 쪽으로 당깁니다 |

계수는 모두 0 이상이고, 두 센서가 겹치는 기간 데이터로 **Poisson 최대우도** 적합합니다.
계수가 하나면 정확히 `MODIS 합 ÷ VIIRS 합`이 되고, 일반적으로도 학습 기간의 합계를 그대로 맞춥니다.
(처음엔 최소제곱으로 적합했는데, 탐지가 많은 칸에 끌려가 합계를 약 15% 적게 잡았습니다.)
LightGBM(03)과 **같은 시험 세트**로 비교해서 어느 쪽을 쓸지 정합니다.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from fsml import evaluate, gbm  # evaluate: 한글 폰트 설정 포함
from fsml.baseline import RatioBaseline
from fsml.features import make_features, training_rows
from fsml.paths import default_grid
from fsml.rules import ModularRuleModel
from fsml.splits import auto_split

GRID = default_grid()
grid = pd.read_parquet(GRID)
res = float(np.round(np.diff(np.sort(grid["lat"].unique())).min(), 6))  # 격자 크기(°)
data, FEATURES = make_features(training_rows(grid), res)
kind, (train, valid, test) = auto_split(data)
fit_rows = train | valid  # 규칙 모델은 조기 종료가 없으니 검증 세트까지 적합에 씁니다
print(f"{GRID.name} | 분할: {kind} | 적합 {fit_rows.sum():,} / 시험 {test.sum():,}")

## 규칙 모델 적합과 계수표

In [ ]:
rules = ModularRuleModel().fit(data[fit_rows])
data["rules"] = rules.predict(data)
rules.table(0)  # 전 세계 계수

**읽는 법:** 계수가 0.05면 "그 구간의 VIIRS 탐지 20건 ≈ MODIS 1건"입니다.
작은 불(<2MW)과 밤의 계수가 작을수록 MODIS가 그런 불을 거의 못 본다는 뜻입니다.
"이웃 칸" 계수는 주변 8칸의 VIIRS 탐지 1건당 이 칸에 더해지는 MODIS 탐지 수입니다.

In [ ]:
rules.table(1)  # 위도대별 계수

## 모듈별 효과

모듈을 하나씩 켜면서 시험 세트 오차가 얼마나 줄어드는지 봅니다. 발표에서 "각 규칙이 왜 필요한가"의 근거가 됩니다.

In [ ]:
variants = {
    "① 비율 1개": ModularRuleModel(use_daynight=False, use_frp=False, use_neighbors=False, levels=()),
    "② + 낮·밤 분리": ModularRuleModel(use_frp=False, use_neighbors=False, levels=()),
    "③ + FRP 구간": ModularRuleModel(use_neighbors=False, levels=()),
    "④ + 주변 칸": ModularRuleModel(levels=()),
    "⑤ + 위도대": ModularRuleModel(levels=(("zone",),)),
    "⑥ + 계절 (전체)": ModularRuleModel(),
}
ablation = {name: m.fit(data[fit_rows]).predict(data[test]) for name, m in variants.items()}
evaluate.compare(data.loc[test, "modis_n"], ablation)

## 기준선 · LightGBM과 비교 (시험 세트)

In [ ]:
data["baseline"] = RatioBaseline().fit(data[fit_rows]).predict(data)
model = gbm.fit(data, FEATURES, train, valid, log_every=0)
data["lightgbm"] = model.predict(data[FEATURES])
evaluate.compare(data.loc[test, "modis_n"], {
    "규칙 모델": data.loc[test, "rules"],
    "LightGBM": data.loc[test, "lightgbm"],
    "기준선": data.loc[test, "baseline"],
    "VIIRS 그대로": data.loc[test, "viirs_n"],
})

달력은 지역·기간 **합계**를 보여주므로 합계 단위 오차가 더 중요합니다.

In [ ]:
totals = evaluate.aggregate(data[test], ["rules", "lightgbm", "baseline"], by=["period"])
evaluate.plot_series(totals, "시험 세트 기간별 합계")
pd.DataFrame({c: evaluate.metrics(totals["modis_n"], totals[c]) for c in ["rules", "lightgbm", "baseline"]}).T

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
evaluate.plot_pred_vs_true(data.loc[test, "modis_n"], data.loc[test, "rules"], "규칙 모델", axes[0])
evaluate.plot_pred_vs_true(data.loc[test, "modis_n"], data.loc[test, "lightgbm"], "LightGBM", axes[1]);

## 결정 메모 (TODO)

- 실제 과거 데이터(시간 분할)에서 규칙 모델과 LightGBM의 지역·월 합계 오차 차이가 작으면 → 규칙 모델 채택 (설명하기 쉬움)
- 차이가 크면 → LightGBM을 쓰고, 규칙 계수표는 "왜 그런 예측이 나오는가"를 설명하는 자료로 사용
- `prior_rows`(희소 그룹을 상위 그룹에 얼마나 붙일지)를 50 / 200 / 1000으로 바꿔 비교
- 토지 피복 열이 생기면 `levels=(("landcover",), ("landcover", "zone"), ("landcover", "zone", "season"))` 시도